# 20 — Deployment with FastAPI

Serve LangChain chains as REST API endpoints.

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'

In [ ]:
import asyncio
import json
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel

## Server Code Template

This is the FastAPI app you would deploy:

In [ ]:
server_code = """
# server.py — run with: uvicorn server:app --reload
from fastapi import FastAPI
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

app = FastAPI(title="LangChain API", version="1.0.0")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

qa_chain = ChatPromptTemplate.from_template("Answer concisely: {question}") | llm | StrOutputParser()
summarize_chain = ChatPromptTemplate.from_template("Summarise in one sentence: {text}") | llm | StrOutputParser()

class QARequest(BaseModel):
    question: str

class SummarizeRequest(BaseModel):
    text: str

@app.get("/health")
async def health(): return {"status": "ok"}

@app.post("/qa")
async def qa(req: QARequest):
    return {"result": await qa_chain.ainvoke({"question": req.question})}

@app.post("/summarize")
async def summarize(req: SummarizeRequest):
    return {"result": await summarize_chain.ainvoke({"text": req.text})}
"""
print(server_code)

## Simulate API Calls

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
qa_chain = ChatPromptTemplate.from_template("Answer concisely: {question}") | llm | StrOutputParser()
summarize_chain = ChatPromptTemplate.from_template("Summarise in one sentence: {text}") | llm | StrOutputParser()

for q in ["What is LangChain?", "What is FastAPI used for?"]:
    result = await qa_chain.ainvoke({"question": q})
    print(f"POST /qa  question={q}\n  → {result}\n")

text = "FastAPI is a modern, fast web framework for building APIs with Python based on type hints."
result = await summarize_chain.ainvoke({"text": text})
print(f"POST /summarize\n  → {result}")